# ADC Toxicity Classifier — Improved Model
Full pipeline: data loading, LOAOCV evaluation with interaction features, final model training, SHAP analysis.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from src.data_loader import load_data
from src.preprocessor import build_preprocessor, FEATURE_COLS, TARGET_REG, TARGET_CLF, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.train import run_loaocv, RF_REG_PARAMS, RF_CLF_PARAMS, XGB_REG_PARAMS, XGB_CLF_PARAMS

df = load_data('data/TADC_Complete_v3_avec_S.xlsx')
print(f"Dataset shape: {df.shape}")
df[['P','D','H','B','L','E','V','S(payload,organe)','%G\u22653 observ\u00e9','Y binaire (G\u22653 >10%)']].describe()

In [ ]:
print('Class distribution (Y binaire):')
print(df['Y binaire (G\u22653 >10%)'].value_counts())
print('\nADC row counts:')
print(df['ADC'].value_counts())

### 1. LOAOCV Evaluation (Leave-One-ADC-Out)
This measures how well the model generalizes to a completely new ADC.

In [ ]:
reg_results = run_loaocv(df, task='regression')
print(f"Improved Regression LOAOCV \u2014 MAE: {reg_results['mae']:.2f}  RMSE: {reg_results['rmse']:.2f}  R\u00b2: {reg_results['r2']:.3f}")

In [ ]:
clf_results = run_loaocv(df, task='classification')
print(f"Improved Classification LOAOCV \u2014 AUC: {clf_results['auc']:.3f}  F1: {clf_results['f1']:.3f}  Acc: {clf_results['accuracy']:.3f}")

### 2. Final Training on Full Data
Including interaction features: P*D, V*S, E/L.

In [ ]:
# Pre-processing with interaction features
df_final = df.copy()
df_final['P_D'] = df_final['P'] * df_final['D']
df_final['V_S'] = df_final['V'] * df_final['S(payload,organe)']
df_final['E_L'] = df_final['E'] / (df_final['L'] + 1e-6)

all_features = FEATURE_COLS + ['P_D', 'V_S', 'E_L']
numeric_features = NUMERIC_FEATURES + ['P_D', 'V_S', 'E_L']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

X_all = preprocessor.fit_transform(df_final[all_features])
y_reg_all = df_final[TARGET_REG].values.astype(float)
y_clf_all = df_final[TARGET_CLF].values.astype(int)

# Regression (RandomForest)
gs_reg = GridSearchCV(
    RandomForestRegressor(random_state=42),
    RF_REG_PARAMS, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
gs_reg.fit(X_all, y_reg_all)
best_reg = gs_reg.best_estimator_

# Classification (RandomForest with balanced weights)
gs_clf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    RF_CLF_PARAMS, cv=5, scoring='roc_auc', n_jobs=-1
)
gs_clf.fit(X_all, y_clf_all)
best_clf = gs_clf.best_estimator_

joblib.dump(best_reg, 'models/model_regression.pkl')
joblib.dump(best_clf, 'models/model_classification.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')
print('Improved models saved.')

### 3. SHAP Analysis (Interpretability)
Identifying which features (including interactions) drive toxicity.

In [ ]:
explainer_reg = shap.TreeExplainer(best_reg)
shap_values_reg = explainer_reg.shap_values(X_all)

feature_names = (
    list(preprocessor.named_transformers_['num'].get_feature_names_out())
    + list(preprocessor.named_transformers_['cat'].get_feature_names_out())
)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_reg, X_all, feature_names=feature_names, show=True)

In [ ]:
explainer_clf = shap.TreeExplainer(best_clf)
shap_values_clf = explainer_clf.shap_values(X_all)
if isinstance(shap_values_clf, list):
    shap_values_clf = shap_values_clf[1]  # Positive class index

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_clf, X_all, feature_names=feature_names, show=True)